# Load data


In [0]:
# transactions
# Read the CSV files from the Unity Catalog Volume into a DataFrame
df = (spark.read
      .format("csv")
      .option("header", True)
      .option("inferSchema", True)
      .load("/Volumes/workspace/travel/travel_data_volume/transactions/"))

# Replace spaces in column names with underscores
df = df.toDF(*[col.replace(" ", "_") for col in df.columns])

# Write the DataFrame to a Delta table
df.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("workspace.travel.transactions")

In [0]:
# manual_logs
# Read the CSV files from the Unity Catalog Volume into a DataFrame
df = (spark.read
      .format("csv")
      .option("header", True)
      .option("inferSchema", True)
      .load("/Volumes/workspace/travel/travel_data_volume/manual_logs/"))

# Replace spaces in column names with underscores
df = df.toDF(*[col.replace(" ", "_") for col in df.columns])

# Write the DataFrame to a Delta table
df.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("workspace.travel.manual_log")

In [0]:
# flight_logs
# Read the CSV files from the Unity Catalog Volume into a DataFrame
df = (spark.read
      .format("csv")
      .option("header", True)
      .option("inferSchema", True)
      .load("/Volumes/workspace/travel/travel_data_volume/flight_logs/"))

# Replace spaces in column names with underscores
df = df.toDF(*[col.replace(" ", "_") for col in df.columns])

# Write the DataFrame to a Delta table
df.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("workspace.travel.flight_log")

In [0]:
# fitbit heart rate
# Read the CSV files from the Unity Catalog Volume into a DataFrame
df = (spark.read
      .format("csv")
      .option("header", True)
      .option("inferSchema", True)
      .load("/Volumes/workspace/travel/travel_data_volume/fitbit/heart_rate"))

# Replace spaces in column names with underscores
df = df.toDF(*[col.replace(" ", "_") for col in df.columns])

# Write the DataFrame to a Delta table
df.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("workspace.travel.fitbit_heart_rate")

In [0]:
# fitbit steps
# Read the CSV files from the Unity Catalog Volume into a DataFrame
df = (spark.read
      .format("csv")
      .option("header", True)
      .option("inferSchema", True)
      .load("/Volumes/workspace/travel/travel_data_volume/fitbit/steps"))

# Replace spaces in column names with underscores
df = df.toDF(*[col.replace(" ", "_") for col in df.columns])

# Write the DataFrame to a Delta table
df.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("workspace.travel.fitbit_steps")

In [0]:
# fitbit sleep score
# Read the CSV files from the Unity Catalog Volume into a DataFrame
df = (spark.read
      .format("csv")
      .option("header", True)
      .option("inferSchema", True)
      .load("/Volumes/workspace/travel/travel_data_volume/fitbit/sleep_score/"))

# Replace spaces in column names with underscores
df = df.toDF(*[col.replace(" ", "_") for col in df.columns])

# Write the DataFrame to a Delta table
df.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("workspace.travel.fitbit_sleep_score")

In [0]:
# google_timeline
# Read the JSON files from the Unity Catalog Volume into a DataFrame
raw_df = (spark.read
          .format("json")
          .option("multiLine", True)
          .load("/Volumes/workspace/travel/travel_data_volume/google_timeline/"))

from pyspark.sql.functions import col
from pyspark.sql.types import StructType, ArrayType

def flatten_df(df):
    flat_cols = []
    nested_cols = []
    for field in df.schema.fields:
        if isinstance(field.dataType, StructType):
            nested_cols.append(field.name)
        elif isinstance(field.dataType, ArrayType):
            nested_cols.append(field.name)
        else:
            flat_cols.append(field.name)
    flat_df = df.select(*flat_cols, *[col(n) for n in nested_cols])
    for n in nested_cols:
        if isinstance(df.schema[n].dataType, ArrayType):
            flat_df = flat_df.withColumn(n, explode_outer(col(n)))
        if isinstance(df.schema[n].dataType, StructType):
            for subfield in df.schema[n].dataType.fields:
                flat_df = flat_df.withColumn(f"{n}_{subfield.name}", col(f"{n}.{subfield.name}"))
            flat_df = flat_df.drop(n)
    return flat_df

df = raw_df
while any(isinstance(field.dataType, (StructType, ArrayType)) for field in df.schema.fields):
    df = flatten_df(df)

# Replace spaces in column names with underscores
df = df.toDF(*[col.replace(" ", "_") for col in df.columns])

# Write the DataFrame to a Delta table
df.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("workspace.travel.google_timeline")

In [0]:
# Check data
display(spark.table("workspace.travel.google_timeline").count())
# display(spark.table("workspace.travel.google_timeline").orderBy(df.columns[0], ascending=False).limit(1000))

In [0]:
# Drop tables
# spark.sql("DROP TABLE IF EXISTS workspace.travel.transactions")
# spark.sql("DROP TABLE IF EXISTS workspace.travel.manual_log")
# spark.sql("DROP TABLE IF EXISTS workspace.travel.flight_log")
# spark.sql("DROP TABLE IF EXISTS workspace.travel.fitbit_heart_rate")
# spark.sql("DROP TABLE IF EXISTS workspace.travel.fitbit_steps")
# spark.sql("DROP TABLE IF EXISTS workspace.travel.fitbit_sleep_score")
spark.sql("DROP TABLE IF EXISTS workspace.travel.google_timeline")